<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 2 (Martes) — Clasificación 1 — Árboles de decisión y bosques aleatorios**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🩺 Árbol de decisión: diagnóstico de tumores (cáncer de mama)

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Clasificar tumores como benignos o malignos a partir de características de células. |
| 🔍 Comprensión de los datos | Cargamos `bcw.csv` (699 observaciones, 9 variables citológicas), revisamos faltantes y la distribución de clases. |
| 🧹 Preparación de los datos | Convertimos la clase a categoría, tratamos nulos y dividimos en entrenamiento (70 %) y prueba (30 %). |
| 🤖 Modelado | Entrenamos un árbol (`criterion="gini"`, `max_depth=3`), lo dibujamos y vemos la importancia de variables. |
| 📊 Evaluación | Matriz de confusión, exactitud, sensibilidad, especificidad y curva ROC. |
| 🚀 Despliegue y conclusiones | Conclusiones clínicas: qué errores son más costosos y cómo mejorar el modelo. |

> 🎯 **Analogía:** un médico que hace preguntas en cadena —¿tamaño del núcleo?, ¿uniformidad celular?— hasta decidir si el tumor es benigno o maligno.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 👉 **2** | Martes | **Clasificación 1: árboles de decisión, bosques aleatorios y KNN** |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('bcw.csv', 'dia_3')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

---
## 🎯 Fase 1 · Comprensión del negocio

Clasificar tumores como benignos o malignos a partir de características de células.

📚 Lectura recomendada: [3]

## Análisis del Problema

Uno de los principales retos que ha encontrado la humanidad de la era moderna, ha sido la lucha contra el cáncer. Solo en el 2012, se le atribuyeron 8.2 millones de muertes a nivel mundial. Los cánceres que causan más muertes son los de pulmón, hígado, estómago, colon y mama. (Fuente: http://www.who.int/cancer/about/facts/es/)

El cáncer de mama se da principalmente en las mujeres, sin embargo también puede darse en los hombres. Como con los otros tipos de cáncer, no se conoce cómo o por qué se desarrolla, a pesar de que se conocen varios factores generales de riesgo (índice de masa corporal elevado, consumo insuficiente de frutas y erduras, falta de actividad física y consumo de tabaco y alcohol.)

A través de los años se han trabajado diferentes enfoques en la lucha contra el cáncer, no de ellos es el de detección temprana. Este enfoque busca detectar los síntomas de la enfermedad apenas están comenzando a desarrollarse, para determinar cómo puede evolucionar y así determinar el mejor tratamiento para el o la paciente.

Sobre esta línea, al detectarse los primeros tumores o masas en el cuerpo de un paciente, una de las principales decisiones es determinar si es un tumor benigno o maligno. Como parte de los esfuerzos para entender mejor esta enfermedad, se realizó un estudio de varias características del núcleo de las células en tumores extraídos de pacientes femeninas en Estados Unidos. El propósito de dicho estudio es tratar de predecir, o clasificar automáticamente si un tumor es benigno o maligno basado en dichas carácterísticas del núcleo de sus células.

---
## 🔍 Fase 2 · Comprensión de los datos

Cargamos `bcw.csv` (699 observaciones, 9 variables citológicas), revisamos faltantes y la distribución de clases.

📚 Lectura recomendada: [4]


## Entendimiento de los Datos

Como parte del estudio, se recolectaron los siguientes datos:

- Sample.number: número de muestra; números enteros entre 61634 y 13454352.
- Thickness: grosor de la masa; números enteros entre 1 y 10.
- Uniformity.Size: uniformidad del tamaño de la célula; números enteros entre 1 y 10.
- Uniformity.Shape: uniformidad de la forma de la célula; números enteros entre 1 y 10.
- Adhesion: adhesión marginal; números enteros entre 1 y 10.
- Epithelial.Size: tamaño de la célula epitelial; números enteros entre 1 y 10.
- Nuclei: nucleo de la célula; números enteros entre 1 y 10.
- Chromatin: cromatina blanda; números enteros entre 1 y 10.
- Nucleoli: "nucleoli" normal; números enteros entre 1 y 10.
- Mitoses: mitosis; números enteros entre 1 y 10.
- Class: clase de tumor; 2 para benigno y 4 para maligno.

## librerías

In [ ]:
import pandas as pd
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import train_test_split

from sklearn.tree import export_graphviz
import graphviz
import plotly.graph_objs as go
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, roc_auc_score

In [ ]:
#cargue el archivo a una variable que se llame bcw usando la función read.csv
#usen el parámetro col.names para pasarle un vector con los nombres de las columnas.
#nombres para las columnas: Sample.number, Thickness, Uniformity.Size, Uniformity.Shape, Adhesion, Epithelial.Size, Nuclei, Chromatin, Nucleoli, Mitoses, Class
#ejemplo: col.names = c('nombre1', 'nombre2')
#usen el parámetro na.strings = '?' para que interprete los signos de pregunta como valores faltantes
bcw = pd.read_csv('sample_data/bcw.csv', header=None, names=['Sample.number', 'Thickness', 'Uniformity.Size', 'Uniformity.Shape', 'Adhesion', 'Epithelial.Size', 'Nuclei', 'Chromatin', 'Nucleoli', 'Mitoses', 'Class'])
# Convertir las columnas de interés a tipo numérico y manejar los errores con 'coerce'
bcwTemp = bcw[['Thickness', 'Uniformity.Size', 'Uniformity.Shape', 'Adhesion', 'Epithelial.Size', 'Nuclei', 'Chromatin', 'Nucleoli', 'Mitoses']] = bcw[['Thickness', 'Uniformity.Size', 'Uniformity.Shape', 'Adhesion', 'Epithelial.Size', 'Nuclei', 'Chromatin', 'Nucleoli', 'Mitoses']].apply(pd.to_numeric, errors='coerce')
# Agregar la columna de la clase a la variable temporal bcwTemp
bcwTemp = pd.concat([bcwTemp, bcw['Class']], axis=1)

In [ ]:
bcwTemp

In [ ]:
bcwTemp.info()

Luego de cargar los datos, podemos dar un primer vistazo a cómo están estructurados:

---
## 🧹 Fase 3 · Preparación de los datos

Convertimos la clase a categoría, tratamos nulos y dividimos en entrenamiento (70 %) y prueba (30 %).

📚 Lectura recomendada: [9]

In [ ]:
#sobreescriban la columna Class con el factor de esa columna
bcwTemp['Class'] = bcwTemp['Class'].astype('category')

#Utilice la función info() para ver la estructura del conjunto de datos:
bcwTemp.info()

In [ ]:
# Chekear valores nulos
print(bcwTemp.isnull().any().sum(), ' / ', len(bcwTemp.columns))

In [ ]:
#Cuales columnnas estas con valores nulos
bcwTemp.isnull().sum()[bcwTemp.isnull().sum() > 0]

In [ ]:
# N/A= Valores vacios
#Usar solo si se quiere eliminar los registros que contienen columnas vacías
bcwTemp=bcwTemp.dropna()

En total, el conjunto de datos con el cual se va a trabajar tiene 699 observaciones, de las cuales 458 son tumores benignos (clase = 2) y 241 son malignos (clase = 4), para una proporción de 65.5% benignos y 34.5% malignos.


---
## 🔍 Fase 2 · Comprensión de los datos — distribución de las clases

Cargamos `bcw.csv` (699 observaciones, 9 variables citológicas), revisamos faltantes y la distribución de clases.

* Distribución de los datos obtenidos

In [ ]:
class_counts = bcwTemp['Class'].value_counts()
print("Distribución de las clases \n",class_counts)

plt.figure()
class_counts.plot.bar(title='Distribución de las clases', ylabel='Observaciones', xlabel='Clase')
plt.show()

---
## 🧹 Fase 3 · Preparación de los datos — partición entrenamiento/prueba

Convertimos la clase a categoría, tratamos nulos y dividimos en entrenamiento (70 %) y prueba (30 %).

Para poder entrenar un modelo de minería de datos, necesitamos dividir el conjunto de datos original en dos diferentes: uno de entrenamiento, con el 70% de las observaciones, y uno de prueba con el restante 30%.

In [ ]:
np.random.seed(4161)

# paso 1: crear el vector lógico
splt = np.random.rand(len(bcwTemp)) < 0.7

# paso 2: crear el data frame de entrenamiento usando los valores TRUE del vector splt
bcw_entrenamiento = bcwTemp[splt]

# paso 3: crear el data frame de prueba negando los valores de splt, para usar las observaciones que en el paso anterior eran falsas
bcw_prueba = bcwTemp[~splt]

Luego de hacer la división, podemos confirmar que no perdimos información y que mantuvimos la proporción del conjunto de datos original:

In [ ]:
#Utilicen la función nrow() para demostrar que en total seguimos trabajando con 699 registros aunque ahora tengamos 2 datasets.
#print((bcw_entrenamiento + bcw_prueba).shape[0])

print("Total de registros entre entrenamiento y pruebas \n",len(pd.concat([bcw_entrenamiento, bcw_prueba])))
print("#################################################")
class_counts = bcw_entrenamiento['Class'].value_counts()
print("Distribución de las clases en entrenamiento \n ",class_counts)
print("#################################################")
class_counts = bcw_prueba['Class'].value_counts()
print("Distribución de las clases en prueba \n ",class_counts)

El conjunto de datos de entrenamiento tiene 321 tumores benignos y 169 malignos (65.5% - 34.5%), y el conjunto de datos de prueba tiene 137 tumores benignos y 72 malignos (65.5% - 34.5%), para un total de 699 observaciones entre ambos).


In [ ]:
# Generar dos gráficos de barra usando plot.bar() para demostrar que se mantiene (o es similar) la proporción de clase = 2 y clase = 4 en los 2 datasets
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
bcw_entrenamiento['Class'].value_counts().plot(kind='bar')
plt.title('Distribución de las clases en entrenamiento')
plt.xlabel('Clase')
plt.ylabel('Observaciones')

plt.subplot(1, 2, 2)
bcw_prueba['Class'].value_counts().plot(kind='bar')
plt.title('Distribución de las clases en prueba')
plt.xlabel('Clase')
plt.ylabel('Observaciones')

plt.tight_layout()
plt.show()


---
## 🤖 Fase 4 · Modelado

Entrenamos un árbol (`criterion="gini"`, `max_depth=3`), lo dibujamos y vemos la importancia de variables.

📚 Lectura recomendada: [5] [10]

![Estructura de un árbol de decisión](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_17.png)
*Figura: nodos internos, ramas y hojas; el criterio (Gini o entropía) decide qué pregunta hacer en cada nodo. [11]*

## Modelo de IA

A partir de los conjuntos de datos de prueba y entrenamiento, vamos a crear un modelo de minería de datos que utilice un árbol de decisión para clasificar observaciones una de las dos clases: 2 (benigno) o 4 (maligno).

Primero, utilizamos el conjunto de datos de entrenamiento para crear el modelo:

In [ ]:
# Crear el modelo
modelo_arbol = DecisionTreeClassifier(criterion='gini', max_depth=3)


# Entrenar el modelo
modelo_arbol.fit(bcw_entrenamiento.iloc[:, :-1], bcw_entrenamiento.iloc[:, -1])


Una vez creado el modelo, procedemos a hacer las predicciones de las classes en nuestro conjunto de datos de prueba, y almacenamos dichas predicciones en un vector llamado "predicciones".

In [ ]:
column_names = bcw.columns
print(column_names)

In [ ]:
dot_data = export_graphviz(modelo_arbol, out_file=None,
                           feature_names=['Thickness', 'Uniformity.Size', 'Uniformity.Shape', 'Adhesion', 'Epithelial.Size', 'Nuclei', 'Chromatin', 'Nucleoli', 'Mitoses'],
                           class_names=['2','4'], filled=True, rounded=True, special_characters=True)
graph = graphviz.Source(dot_data)
graph

In [ ]:
# Obtener la importancia de cada variable
importancias = modelo_arbol.feature_importances_

# Crear un DataFrame con los nombres de las variables y sus respectivas importancias
df_importancias = pd.DataFrame({'Variable': ['Thickness', 'Uniformity.Size', 'Uniformity.Shape', 'Adhesion', 'Epithelial.Size', 'Nuclei', 'Chromatin', 'Nucleoli', 'Mitoses'], 'Importancia': importancias})

# Ordenar el DataFrame de forma descendente según la importancia de las variables
df_importancias = df_importancias.sort_values('Importancia', ascending=False)

# Imprimir el DataFrame resultante
print(df_importancias)


# Creamos una lista con los nombres de las variables y sus importancias
var_imp = list(zip(bcw.columns[:-1], modelo_arbol.feature_importances_))
# Ordenamos la lista por importancia en orden ascendente
var_imp.sort(key=lambda x: x[1])

# Creamos un objeto de barras verticales
fig = go.Figure(data=[go.Bar(x=[x[1] for x in var_imp], y=[x[0] for x in var_imp], orientation='h')])

# Definimos el título del gráfico y los nombres de los ejes
fig.update_layout(title='Importancia de las variables', xaxis_title='Importancia', yaxis_title='Variable')

# Mostramos el gráfico
fig.show()

---
## 📊 Fase 5 · Evaluación

Matriz de confusión, exactitud, sensibilidad, especificidad y curva ROC.

📚 Lectura recomendada: [7] [8]

## Evaluación

Para evaluar la eficacion del modelo de minería de datos, podemos tabular las predicciones (columnas) con los valores reales del conjunto de prueba (hileras):

*Notas Importantes*:

![Matriz de confusión: VP, FN, FP y VN](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_08.png)
*Figura: Matriz de confusión: VP, FN, FP y VN. [11]*

![Ejemplo de matriz de confusión con conteos reales](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_09.png)
*Figura: Ejemplo de matriz de confusión con conteos reales. [11]*

In [ ]:
#En este ejemplo, X_prueba es la matriz de características del conjunto de datos de prueba,
# y_prueba es el vector de valores de la variable objetivo en el conjunto de datos de prueba,
# y modelo_arbol es el modelo de árbol de decisión que has entrenado. La función confusion_matrix
# devuelve una matriz de confusión que muestra cuántas observaciones fueron clasificadas correctamente y
# cuántas fueron clasificadas incorrectamente por el modelo.

predicciones = modelo_arbol.predict(bcw_prueba.iloc[:, :-1])
matriz_confusion = confusion_matrix(bcw_prueba.iloc[:, -1], predicciones)
print(matriz_confusion)

In [ ]:
# Crear el gráfico de la matriz de confusión
fig, ax = plt.subplots()
im = ax.matshow(matriz_confusion, cmap=plt.cm.Blues)

# Agregar los valores de cada celda en el gráfico
for i in range(matriz_confusion.shape[0]):
    for j in range(matriz_confusion.shape[1]):
        ax.text(x=j, y=i, s=matriz_confusion[i, j], va='center', ha='center')


plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.xticks([0,1], ['Benigno', 'Maligno'])
plt.yticks([0,1], ['Benigno', 'Maligno'])
plt.title('Matriz de Confusión')
# Agregar la barra de color
plt.colorbar(im)
plt.show()

En general, podemos sacar las siguientes métricas a partir de la tabulación anterior:

\begin{equation}Exactitud=\frac{VP+VN}{Total}\end{equation}

- *Exactitud Total* (observaciones clasificadas apropiadamente) = 93.78% (128 benignos y 68 malignos clasificados correctamente).

In [ ]:
print("Precisión global:", (matriz_confusion[0, 0] + matriz_confusion[1, 1]) / sum(sum(matriz_confusion)))

\begin{equation}\text{Sensibilidad}=\frac{VP}{\text{Total Positivos}}\end{equation}
- *Sensibilidad* (procentaje de positivos verdaderos - de las observaciones que realmente son benignas, ¿cuántas clasificó apropiadamente el modelo?) = 93.42% (de las 137 observaciones que son benignos, 128 fueron correctamente identificadas).

In [ ]:
print("Sensibilidad:", matriz_confusion[1, 1] / (matriz_confusion[1, 1] + matriz_confusion[1, 0]))

\begin{equation}\text{Especificidad}=\frac{VN}{\text{Total Negativos}}\end{equation}
- *Especificidad* (porcentaje de negativos verdaderos - en este caso, de las observaciones que realmente son malignas, ¿cuántas clasificó apropiadamente el modelo?) = 94.44% (de las 72 observaciones que son malignas, 68 fueron correctamente identificadas).

In [ ]:
print("Especificidad:", matriz_confusion[0, 0] / (matriz_confusion[0, 0] + matriz_confusion[0, 1]))

In [ ]:
# Calcular el score ROC y la curva ROC

y_true = (bcw_prueba.iloc[:, -1] == 4).astype(int)

score_roc = roc_auc_score(y_true, predicciones)

print("El score es", score_roc)

fpr, tpr, umbrales = roc_curve(y_true, predicciones)

# Generar el gráfico de la curva ROC
plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % score_roc)
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC')
plt.legend(loc="lower right")
plt.show()

*Curva ROC: cuanto más se acerca a la esquina superior izquierda, mejor separa el modelo las clases.*

![Curvas ROC con su AUC y la línea de adivinar al azar](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_10.png)
*Figura: Curvas ROC con su AUC y la línea de adivinar al azar. [11]*

---
## 🚀 Fase 6 · Despliegue y conclusiones

Conclusiones clínicas: qué errores son más costosos y cómo mejorar el modelo.

📚 Lectura recomendada: [8]

## Resultados

En general, los resultados del modelo son considerablemente buenos, a pesar de ser un modelo relativamente sencillo. De las observaciones clasificadas, el 94% se hizo apropiadamente.

Con respecto al tipo de decisiones que se pueden tomar en este caso, la decisión que puede ser más dañina para un paciente es que tenga un tumor maligno y erróneamente se diga que es benigno, pues el paciente podría no recibir tratamiento alguno y eventualmente fallecer. La *especificidad* del modelo es del 94.44%, así que podemos decir que el modelo solo tiene alrededor de un 5% de posibilidades de cometer este error.

Alternativamente, se le podría decir al paciente que tiene un tumor maligno cuando en realidad el tumor es benigno. En este caso, el paciente podría pasar un mal rato emocionalmente y puede ser sujeto de más pruebas que eventualmente mostrarían la verdad. La *sensitividad* del modelo es del 93.42%, lo cual quiere decir que el modelo tiene cerca de un 7% probabilidad de cometer este error. Asimismo, la *precisión* es muy alta cerca del 97%, por lo que queda al criterio de los tomadores de decisiones usar o no este modelo.

---
## 🧪 Refuerzo: elegir la profundidad con validación cruzada (poda por hiperparámetros)

Un árbol muy profundo sobreajusta; uno muy corto, subajusta. En lugar de elegir `max_depth` a ojo, se busca con **validación cruzada** (`GridSearchCV`), que prueba cada valor en varias particiones del entrenamiento [9]. Además, en diagnóstico médico importa más la **sensibilidad** (no dejar escapar tumores malignos) que la exactitud global [8].

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

X_tr, y_tr = bcw_entrenamiento.iloc[:, :-1], bcw_entrenamiento.iloc[:, -1]
X_te, y_te = bcw_prueba.iloc[:, :-1], bcw_prueba.iloc[:, -1]

gs = GridSearchCV(DecisionTreeClassifier(random_state=0),
                  {'max_depth': [1, 2, 3, 4, 5, 6, 8], 'min_samples_leaf': [1, 3, 5, 10]},
                  cv=5, scoring='recall_macro').fit(X_tr, y_tr)
print('Mejores parámetros:', gs.best_params_, '| recall medio en validación cruzada:', round(gs.best_score_, 3))
print(classification_report(y_te, gs.best_estimator_.predict(X_te)))

### ✍️ Ejercicios
1. ¿Coincide el `max_depth` elegido con el 3 usado arriba? ¿Mejora el *recall* de la clase maligna?
2. Cambie `scoring` a `'accuracy'`: ¿se elige el mismo árbol?

---
## ✅ Conclusiones y siguientes pasos

En medicina, un **falso negativo** (tumor maligno clasificado como benigno) es mucho más grave que un falso positivo, por eso se vigila la **sensibilidad** [8]. Un árbol de profundidad 3 ya da buen desempeño y es explicable para un clínico.

---
## 📚 Referencias

Fuentes adicionales para profundizar: [6].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Wolberg, W. H., & Mangasarian, O. L. (1990). Multisurface method of pattern separation for medical diagnosis applied to breast cytology. *Proceedings of the National Academy of Sciences, 87*(23), 9193–9196. https://doi.org/10.1073/pnas.87.23.9193

**[4]** Dua, D., & Graff, C. (2019). *UCI Machine Learning Repository*. University of California, Irvine. https://archive.ics.uci.edu

**[5]** Breiman, L., Friedman, J. H., Olshen, R. A., & Stone, C. J. (1984). *Classification and Regression Trees*. Wadsworth.

**[6]** Quinlan, J. R. (1986). Induction of decision trees. *Machine Learning, 1*(1), 81–106. https://doi.org/10.1007/BF00116251

**[7]** Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters, 27*(8), 861–874. https://doi.org/10.1016/j.patrec.2005.10.010

**[8]** Powers, D. M. W. (2011). Evaluation: From precision, recall and F-measure to ROC, informedness, markedness and correlation. *Journal of Machine Learning Technologies, 2*(1), 37–63.

**[9]** James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*. Springer. https://doi.org/10.1007/978-3-031-38747-0

**[10]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

**[11]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)